添加一个新的预估命令，这个命令就是能在原有的基数估计上实现，带推理谓词的基数估计，先只考虑树采样，对于满足结构匹配的结果，再经过一次条件检查:只有对应post节点的oracle列满足>0.5，才会success++。对于post的oracle调用的结果已经保存到post.csv文件中，可以预先缓存，通过hash查找的方式直接查找满足子图匹配嵌入中推理节点的oracle结果。


In [2]:
%load_ext autoreload
%autoreload 2

In [1]:
import os
import re
import sys
import math
import time
import tempfile
import traceback
from typing import List, Dict, Tuple
import pandas as pd
import numpy as np
import json
project_root = "/home/wangshuo/projects/Neo4j_Exp"
if project_root not in sys.path:
    sys.path.append(project_root)
from pythonProject.src.Structure_first.fastest_pipeline import FastestGraphConverter, FastestEstimateMerger
from pythonProject.src.Structure_first.graph_sample import FastestRunner
from pythonProject.src.Structure_first.precision_submatching import ExactSubgraphMatcher
from pythonProject.src.Structure_first.proxy_sample import ProxyStratifiedSampler, compute_T_true

# 一级测试数据集
datasets_name = "parler_data"
# 一级数据集下根据查询和图结构的差异划分的子测试数据集
dataset_name = "dataset_three"
# dataset_name = "dataset_one"
# 原始CSV数据路径
CSV_BASE_DIR = f"/home/wangshuo/resource/datasets/{datasets_name}/{dataset_name}/csv_data"
# 转换后GraphLib数据存放路径
Graph_Lib_Dir = f"/home/wangshuo/resource/datasets/{datasets_name}/{dataset_name}/data_graph"

# converter = FastestGraphConverter(CSV_BASE_DIR,Graph_Lib_Dir)
# converter.run_without_author_user_post()
# converter.remove_edge_labels()

In [2]:
# 初始化 Runner
runner = FastestRunner(build_dir="/home/wangshuo/projects/FaSTest-main/build")

# dataset_name = "dataset_three"  # ✅ 在这里修改为你的数据集名
current_budget = 20000

# [重要] 这里即便指定了 root_label，如果 C++ 端读取到了 user_custom_labels 配置，
# 会优先使用配置中的多标签（post & comment）。
# 但为了兼容性和程序健壮性，通常仍给一个有效的 root label。
infer_label = 3  

# 定义各个 CSV 中的 Oracle 列名
post_col = "ML1_oracle2_probability"         # Post 表里的概率列
comment_col = "ML2_oracle2_probability"      # Comment 表里的概率列（根据实际情况修改）
proxy_folder = "ML1_proxy4b_probability_ML2_proxy1_probability"

# 调用 run 方法
code, output = runner.run(
    dataset=dataset_name,
    root_label=infer_label,
    sample_budget=current_budget,
    estimate_with_predicate=True,            # <--- 必须开启
    post_oracle_col=post_col,                # <--- 传递 Post 列名
    comment_oracle_col=comment_col,           # <--- 传递 Comment 列名
    multi_proxy_prob=proxy_folder
)

🚀 正在运行: /home/wangshuo/projects/FaSTest-main/build/Fastest -d dataset_three --ROOT_LABEL 3 --SAMPLE_BUDGET 20000 --ESTIMATE_WITH_PREDICATE --POST_ORACLE_COL ML1_oracle2_probability --COMMENT_ORACLE_COL ML2_oracle2_probability --MULTI_PROXY_PROB ML1_proxy4b_probability_ML2_proxy1_probability

[CLI] Root label set to 3
[CLI] Sample budget set to 20000
[CLI] Mode: Estimate with Predicate enabled.
[CLI] Post Oracle Col: ML1_oracle2_probability
[CLI] Comment Oracle Col: ML2_oracle2_probability
[CLI] Multi Proxy Prob Dir Name: ML1_proxy4b_probability_ML2_proxy1_probability
[info] SAMPLE_BUDGET:20000
[Info] Cleared old estimate file: /home/wangshuo/resource/datasets/parler_data/dataset_three/results/in_estimateW_result.txt
[info] args:0x7ffce38255f8
[info] summary_run1_path:/home/wangshuo/resource/datasets/parler_data/dataset_three/results/result_summarys/ML1_proxy4b_probability_ML2_proxy1_probability/results_summary_run_1.csv
/home/wangshuo/resource/datasets/parler_data/dataset_three/ground_

KeyboardInterrupt: 